In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from tabicl import TabICLClassifier

df = pd.read_csv("sample_customers.csv")
df.head()

/Users/zhl/Desktop/tabular-inference-prototype/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,age,income,balance,tenure_years,region,default
0,60,49592,1639,8,BC,0
1,67,48434,44348,17,ON,0
2,22,38905,6140,6,BC,0
3,21,108563,13031,20,QC,0
4,57,71463,425,5,QC,0


In [3]:
X = df.drop(columns=["default"])
y = df["default"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42
)

model = TabICLClassifier()

model.fit(X_train, y_train)

pred = model.predict(X_test)

print(pred[:10])

INFO: You are downloading 'tabicl-classifier-v2-20260212.ckpt', the latest best-performing version, used in our TabICLv2 paper.

Checkpoint 'tabicl-classifier-v2-20260212.ckpt' not cached.

[0 0 0 0 0 0 0 0 0 0]


In [4]:
print(y_test.iloc[:10].to_numpy())

[0 0 0 0 1 0 0 0 0 0]


In [5]:
print(y.value_counts())
print(y.value_counts(normalize=True))

default
0    357
1     43
Name: count, dtype: int64
default
0    0.8925
1    0.1075
Name: proportion, dtype: float64


In [6]:
from sklearn.metrics import classification_report

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.88      1.00      0.93        70
           1       0.00      0.00      0.00        10

    accuracy                           0.88        80
   macro avg       0.44      0.50      0.47        80
weighted avg       0.77      0.88      0.82        80



/Users/zhl/Desktop/tabular-inference-prototype/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/zhl/Desktop/tabular-inference-prototype/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/zhl/Desktop/tabular-inference-prototype/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control thi

In [7]:
proba = model.predict_proba(X_test)[:, 1]

print(proba[:20])
print("max probability:", proba.max())

[0.10996574 0.02703819 0.04710092 0.12298916 0.27993095 0.0436457
 0.04588541 0.09664    0.06343413 0.05892403 0.03738667 0.14036985
 0.02777728 0.02442741 0.02219109 0.10871455 0.07460798 0.06158346
 0.27623537 0.04175052]
max probability: 0.28091872


In [8]:
from sklearn.metrics import roc_auc_score

print("AUC:", roc_auc_score(y_test, proba))

AUC: 0.7542857142857142


In [9]:
threshold = 0.15

pred_custom = (proba >= threshold).astype(int)

print(classification_report(y_test, pred_custom))

              precision    recall  f1-score   support

           0       0.94      0.90      0.92        70
           1       0.46      0.60      0.52        10

    accuracy                           0.86        80
   macro avg       0.70      0.75      0.72        80
weighted avg       0.88      0.86      0.87        80

